In [1]:
import tensorflow as tf

@tf.function
def SoftSandNew(Kf, RHOf, Phi, Quartz, Clay, Feldspar, Limestone, Dolomite, Pressure, PhiC, Coordination, Fudge):
    # Convertir todos los parámetros a float32
    Kf = tf.cast(Kf, tf.float32)
    RHOf = tf.cast(RHOf, tf.float32)
    Phi = tf.cast(Phi, tf.float32)
    Quartz = tf.cast(Quartz, tf.float32)
    Clay = tf.cast(Clay, tf.float32)
    Feldspar = tf.cast(Feldspar, tf.float32)
    Limestone = tf.cast(Limestone, tf.float32)
    Dolomite = tf.cast(Dolomite, tf.float32)
    Pressure = tf.cast(Pressure, tf.float32)
    PhiC = tf.cast(PhiC, tf.float32)
    Coordination = tf.cast(Coordination, tf.float32)
    Fudge = tf.cast(Fudge, tf.float32)

    # Resto de la función
    Dolomite = 1 - (Quartz + Clay + Feldspar + Limestone)

    RHOClay = 2.65 
    KClay = 21 
    GClay = 7
    KsV = Quartz * 36.6 + Clay * KClay + Limestone * 76.8 + Dolomite * 94.9 + Feldspar * 75.6
    KsR = 1 / (Quartz / 36.6 + Clay / KClay + Limestone / 76.8 + Dolomite / 94.9 + Feldspar / 75.6)
    Ks = 0.5 * (KsV + KsR)
    
    GsV = Quartz * 45 + Clay * GClay + Limestone * 32 + Dolomite * 45 + Feldspar * 25.6
    GsR = 1 / (Quartz / 45 + Clay / GClay + Limestone / 32 + Dolomite / 45 + Feldspar / 25.6)
    Gs = 0.5 * (GsV + GsR)
    Ms = Ks + (4 / 3) * Gs
    NUs = 0.5 * (Ms / Gs - 2) / (Ms / Gs - 1)
    
    RHOs = Quartz * 2.65 + Clay * RHOClay + Limestone * 2.71 + Dolomite * 2.87 + Feldspar * 2.63
    
    P = Pressure / 1000
    C = Coordination
    
    a = tf.pow(((3 * 3.14 * (1 - NUs) / (2 * C * (1 - PhiC) * Gs)) * P), (1 / 3))
    SN = 4 * a * Gs / (1 - NUs)
    ST = 8 * a * Gs / (2 - NUs)
    ST = ST / Fudge
    
    Khat = C * (1. - PhiC) * SN / (12 * 3.14)
    Ghat = C * (1. - PhiC) * (SN + 1.5 * ST) / (20 * 3.14)
    
    KDry1 = 1 / ((Phi / PhiC) / (Khat + 4 * Ghat / 3) + ((PhiC - Phi) / PhiC) / (Ks + 4 * Ghat / 3)) - 4 * Ghat / 3
    ZZ1 = (Ghat / 6) * (9 * Khat + 8 * Ghat) / (Khat + 2 * Ghat)
    GDry1 = 1 / ((Phi / PhiC) / (Ghat + ZZ1) + ((PhiC - Phi) / PhiC) / (Gs + ZZ1)) - ZZ1
    MDry1 = KDry1 + (4 / 3) * GDry1
    NuDry1 = 0.5 * (MDry1 / GDry1 - 2) / (MDry1 / GDry1 - 1)
    
    KDry2 = 1 / (((1 - Phi) / (1 - PhiC)) / (Khat + 4 * Ghat / 3) + ((Phi - PhiC) / (1 - PhiC)) / (4 * Ghat / 3)) - 4 * Ghat / 3
    ZZ2 = (Ghat / 6) * (9 * Khat + 8 * Ghat) / (Khat + 2 * Ghat)
    GDry2 = 1 / (((1 - Phi) / (1 - PhiC)) / (Ghat + ZZ2) + ((Phi - PhiC) / (1 - PhiC)) / (ZZ2)) - ZZ2
    MDry2 = KDry2 + (4 / 3) * GDry2
    NuDry2 = 0.5 * (MDry2 / GDry2 - 2) / (MDry2 / GDry2 - 1)
    
    MDry = MDry1 * tf.cast((Phi <= PhiC), tf.float32) + MDry2 * tf.cast((Phi > PhiC), tf.float32)
    GDry = GDry1 * tf.cast((Phi <= PhiC), tf.float32) + GDry2 * tf.cast((Phi > PhiC), tf.float32)
    NuDry = NuDry1 * tf.cast((Phi <= PhiC), tf.float32) + NuDry2 * tf.cast((Phi > PhiC), tf.float32)
    KDry = MDry - (4 / 3) * GDry
    
    KSat = Ks * (Phi * KDry - (1 + Phi) * Kf * KDry / Ks + Kf) / ((1 - Phi) * Kf + Phi * Ks - Kf * KDry / Ks)
    MSat = KSat + (4 / 3) * GDry
    RHOB = (1 - Phi) * RHOs + Phi * RHOf
    Vp = tf.pow((MSat / RHOB), 0.5)
    Vs = tf.pow((GDry / RHOB), 0.5)
    Ip = Vp * RHOB
    PR = 0.5 * (MSat / GDry - 2) / (MSat / GDry - 1)
    
    return Vp, Vs, RHOB

# Definir el tamaño del batch
batch_size = 10
input_shape = [124, 124]

# Generar el tensor de entrada con valores aleatorios entre 0 y 0.3
random_tensor = tf.random.uniform(shape=input_shape, minval=0, maxval=0.3, dtype=tf.float32)

# Imprimir el tensor generado para verificar que tiene los valores esperados
print("Random Tensor:")
print(random_tensor.numpy())

# Separar las entradas según lo necesite la función
# Aquí se asume que las primeras dimensiones del tensor aleatorio corresponden a los diferentes parámetros que necesita la función
Phi = random_tensor[:, 0]
Quartz = random_tensor[:, 1]
Clay = random_tensor[:, 2]
Feldspar = random_tensor[:, 3]
Limestone = random_tensor[:, 4]
Dolomite = random_tensor[:, 5]  # o calcula 1 - (Quartz + Clay + Feldspar + Limestone) si es necesario

# Imprimir las entradas para verificar
print("Phi:", Phi.numpy())
print("Quartz:", Quartz.numpy())
print("Clay:", Clay.numpy())
print("Feldspar:", Feldspar.numpy())
print("Limestone:", Limestone.numpy())
print("Dolomite:", Dolomite.numpy())

# Definir otros parámetros constantes
Kf = tf.constant(1.0, dtype=tf.float32)
RHOf = tf.constant(1.0, dtype=tf.float32)
Pressure = tf.constant(10.0, dtype=tf.float32)
PhiC = tf.constant(0.4, dtype=tf.float32)
Coordination = tf.constant(6.0, dtype=tf.float32)
Fudge = tf.constant(1.0, dtype=tf.float32)

# Llamar a la función y obtener los resultados
Vp_final, Vs_final, RHO_final = SoftSandNew(Kf, RHOf, Phi, Quartz, Clay, Feldspar, Limestone, Dolomite, Pressure, PhiC, Coordination, Fudge)

# Mostrar los resultados
print("Vp_final:")
print(Vp_final.numpy())

print("Vs_final:")
print(Vs_final.numpy())

print("RHO_final:")
print(RHO_final.numpy())


2024-05-27 17:11:04.320042: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-27 17:11:04.320098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-27 17:11:04.321067: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-27 17:11:04.327275: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-27 17:11:05.208530: W tensorflow/compiler/tf2

Random Tensor:
[[0.08580795 0.27485442 0.00355636 ... 0.03443985 0.08211254 0.1569947 ]
 [0.04202625 0.11390258 0.14349215 ... 0.05055989 0.14653331 0.22843027]
 [0.23875831 0.10582427 0.0318694  ... 0.2658768  0.04449113 0.13343897]
 ...
 [0.0027876  0.03294225 0.00414752 ... 0.03964237 0.18271351 0.10974652]
 [0.09249526 0.12578331 0.15548891 ... 0.04161791 0.07278189 0.06242756]
 [0.2626864  0.14953734 0.11101077 ... 0.21177639 0.00676621 0.23406906]]
Phi: [0.08580795 0.04202625 0.23875831 0.22107078 0.16631353 0.03465708
 0.1640884  0.15145862 0.11342612 0.13457899 0.17105632 0.21705431
 0.1444085  0.17532274 0.25318152 0.11856412 0.21318766 0.15438907
 0.20757316 0.28353333 0.2565692  0.22008903 0.11761605 0.11990952
 0.08626449 0.11809921 0.15690166 0.26310188 0.2784952  0.10712239
 0.03211888 0.19966634 0.0368272  0.28719243 0.00678996 0.02158678
 0.19503376 0.05855069 0.10444898 0.20594639 0.2760024  0.15320504
 0.07963085 0.28307927 0.2531474  0.05599709 0.08390579 0.05207223


2024-05-27 17:11:07.167637: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Vp_final:
[3.5616024 4.1290393 2.2187524 2.2792342 2.6611657 4.383786  2.6344223
 2.5480032 2.825237  2.7130706 2.389996  2.1296265 2.7746353 2.5207531
 2.079335  2.9413233 2.1584978 2.7699313 2.410379  1.9182419 2.1947548
 2.1772044 2.8317792 2.8570325 3.4191136 3.003132  2.4575431 2.039464
 2.0112982 3.1026714 4.6143427 2.3989713 4.358377  1.8482187 5.63771
 4.563181  2.4572852 3.6991813 3.0975895 2.3313527 1.9966048 2.514511
 3.2547674 2.074377  2.0562437 3.87647   3.2423391 3.8710632 1.9889095
 4.6234903 2.0502524 2.1755555 3.0459878 2.5091681 2.304531  2.0097384
 4.5765347 2.0228462 2.1329417 1.9991665 2.5977519 2.5002291 2.4397864
 5.530845  3.750488  2.501788  1.9374332 2.2474825 3.0388327 2.4383366
 2.5262787 5.667483  2.041006  3.5982463 4.349893  4.638034  2.381145
 1.9842892 3.4858909 2.0618198 2.0705214 2.3465457 1.9516947 2.487663
 3.1886587 1.9646969 2.4061081 3.3262496 2.2362425 4.350929  2.5404372
 2.2573771 1.9265826 2.4713085 2.4712458 2.3066125 2.3976016 4.4576173
 2

In [4]:
Vp_final

<tf.Tensor: shape=(124,), dtype=float32, numpy=
array([3.5616024, 4.1290393, 2.2187524, 2.2792342, 2.6611657, 4.383786 ,
       2.6344223, 2.5480032, 2.825237 , 2.7130706, 2.389996 , 2.1296265,
       2.7746353, 2.5207531, 2.079335 , 2.9413233, 2.1584978, 2.7699313,
       2.410379 , 1.9182419, 2.1947548, 2.1772044, 2.8317792, 2.8570325,
       3.4191136, 3.003132 , 2.4575431, 2.039464 , 2.0112982, 3.1026714,
       4.6143427, 2.3989713, 4.358377 , 1.8482187, 5.63771  , 4.563181 ,
       2.4572852, 3.6991813, 3.0975895, 2.3313527, 1.9966048, 2.514511 ,
       3.2547674, 2.074377 , 2.0562437, 3.87647  , 3.2423391, 3.8710632,
       1.9889095, 4.6234903, 2.0502524, 2.1755555, 3.0459878, 2.5091681,
       2.304531 , 2.0097384, 4.5765347, 2.0228462, 2.1329417, 1.9991665,
       2.5977519, 2.5002291, 2.4397864, 5.530845 , 3.750488 , 2.501788 ,
       1.9374332, 2.2474825, 3.0388327, 2.4383366, 2.5262787, 5.667483 ,
       2.041006 , 3.5982463, 4.349893 , 4.638034 , 2.381145 , 1.9842892,
   